Esta etapa silver toma la data de bronze y mejora la calidad de la información eliminando duplicados con un procesamiento incremental. 

Inferimos el esquema del json

In [0]:
ruta_origen = "iol_challenge.bronze.raw_ingestion"
ruta_destino = "iol_challenge.silver.deduped_transactions"

In [0]:
from pyspark.sql.functions import from_json, schema_of_json, col

data_sample = spark.read \
    .format("delta") \
    .table(ruta_origen) \
    .filter(col("data_type") == "transaction") \
    .withColumn("schema", schema_of_json(col("data"))) \
    .select("schema").first()[0]

In [0]:
data_sample

'STRUCT<cantidad: BIGINT, descripcion_titulo: STRING, fecha: STRING, id_cliente: STRING, id_transaccion: STRING, moneda: STRING, origen: STRING, precio: DOUBLE, simbolo_titulo: STRING, tipoTran: STRING>'

In [0]:
from pyspark.sql.functions import col, row_number, max as spark_max, length, md5, concat_ws, lit, explode, from_json
from pyspark.sql.window import Window
from delta.tables import DeltaTable



schema = """
STRUCT<fecha: STRING, 
tipoTran: STRING,
id_cliente: STRING, 
descripcion_titulo: STRING, 
moneda: STRING, 
simbolo_titulo: STRING, 
cantidad: BIGINT, 
precio: DOUBLE, 
id_transaccion: STRING, 
origen: STRING>
"""

df_origen_raw = spark.read \
    .format("delta") \
    .table(ruta_origen) \
    .filter(col("data_type") == "transaction") \
    .withColumn("parsed_dict", from_json(col("data"), schema)) \
    .withColumn("sk_transaccion", md5(concat_ws(lit("||"), col("parsed_dict.id_transaccion")))) \
    .select(
        col("sk_transaccion"),
        col("parsed_dict.*"), 
        col("dia"),
        col("anio"),
        col("mes"),
        col("timestamp_ejecucion"),
        col("errores_calidad"),
        col("tiene_errores_calidad"),
    )

# Tomamos la última versión ingestada de la transacción preferentemente sin errores de calidad
ventana_dedup = Window.partitionBy("id_transaccion").orderBy(col("tiene_errores_calidad").asc(), col("timestamp_ejecucion").desc())

df_lote_deduplicado = df_origen_raw \
    .withColumn("row_num", row_number().over(ventana_dedup)) \
    .filter(col("row_num") == 1) \
    .drop("row_num") \


if spark.catalog.tableExists(ruta_destino):
    tabla_destino = DeltaTable.forName(spark, ruta_destino)
    
    max_timestamp = tabla_destino.toDF() \
        .select(spark_max("timestamp_ejecucion")) \
        .collect()[0][0]
    
    if max_timestamp is not None:
        df_lote_filtrado = df_lote_deduplicado.filter(col("timestamp_ejecucion") >= max_timestamp)
    else:
        df_lote_filtrado = df_lote_deduplicado

    tabla_destino.alias("target") \
        .merge(
            df_lote_filtrado.alias("source"),
            "target.id_transaccion = source.id_transaccion"
        ) \
        .whenMatchedUpdateAll(
            condition="source.timestamp_ejecucion > target.timestamp_ejecucion"
        ) \
        .whenNotMatchedInsertAll() \
        .execute()

else:
    df_lote_deduplicado.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(ruta_destino)

In [0]:
%sql
SELECT * FROM iol_challenge.silver.deduped_transactions limit 10;


sk_transaccion,fecha,tipoTran,id_cliente,descripcion_titulo,moneda,simbolo_titulo,cantidad,precio,id_transaccion,origen,dia,anio,mes,timestamp_ejecucion,errores_calidad,tiene_errores_calidad
654c56453d1900ac678ac8e27d886e30,2026-01-28T06:47:26.000Z,Venta,CLI323A59A0,"Cedear Netflix, Inc.",ARS,NFLX,3,2691.1216,TXN00130gbuq5cb8,App Mobile,1,2026,1,2026-08-05T05:58:01.342Z,null,false
0527e45976cf28772e0e0818c1b9d49f,2026-02-05T01:34:05.000Z,Venta,CLI0B0868D7,Grupo Financiero Galicia S.A,ARS,GGAL,6,7995.8703,TXN0014pi30pgnh1,App Mobile,2,2026,2,2026-08-05T05:58:01.342Z,null,false
c34448d3914a9d07172aa0969b952ba8,2026-01-30T09:42:23.000Z,Compra,CLIFCE7A53B,Bbva,ARS,BBAR,1,9972.4962,TXN001671nd5iug5,App Mobile,1,2026,1,2026-08-05T05:58:01.342Z,null,false
40adf9a6bb50b043ce76ea5fa1651a92,2026-03-02T19:45:51.000Z,Compra,CLI42C89E5A,Transportadora Gas del Sur,ARS,TGSU2,3,9152.431,TXN001d63us6ohw0,Sitio Web Desktop,3,2026,3,2026-08-05T05:58:01.342Z,null,false
c85d275546bddb8a2d393196a51bcefa,2026-02-02T21:48:01.000Z,Compra,CLIA02ABE02,Bono Rep. Argentina Usd Step Up 2030,ARS,AL30,2,884.6185,TXN001whi1fl1omy,App Mobile,2,2026,2,2026-08-05T05:58:01.342Z,null,false
79e2af93f8bc7dadf331ec00fc0706e5,2026-02-09T19:08:49.000Z,Compra,CLI1F89EF2D,Cedear The Coca Cola Company,ARS,KO,2,23505.5037,TXN0023q8749xe1v,App Mobile,2,2026,2,2026-08-05T05:58:01.342Z,null,false
b7523bec29ae3f89ffb74736533cac71,2026-03-13T05:02:21.000Z,Venta,CLI42F444C8,Cedear Spdr S&P 500,ARS,SPY,2,49040.4264,TXN0028manmlvlj5,App Mobile,3,2026,3,2026-08-05T05:58:01.342Z,null,false
aa1826316d7c6f0d261711b36418c7f6,2026-01-14T19:35:22.000Z,Compra,CLI36A35DBC,Cedear Microsoft Corp.,ARS,MSFT,2,23521.2603,TXN00395gzt3a24z,Sitio Web Responsive,1,2026,1,2026-08-05T05:58:01.342Z,null,false
0a1dd4eeb2ba62a88704f65fa14afd3d,2026-01-13T19:51:18.000Z,Compra,CLIF0893C5A,Cedear Taiwan Semic. Manuf.,ARS,TSM,1,56512.0619,TXN004ov11xzrlff,App Mobile,1,2026,1,2026-08-05T05:58:01.342Z,null,false
a30a444864a743d0a93a295028eb3eb5,2026-02-04T02:50:41.000Z,Venta,CLI3FC35AB3,Aluar,ARS,ALUA,10,980.8941,TXN004ru35okfz0t,Sitio Web Desktop,2,2026,2,2026-08-05T05:58:01.342Z,null,false


In [0]:
%sql
SELECT id_transaccion, COUNT(*) FROM iol_challenge.silver.deduped_transactions group by id_transaccion
    having count(*)>1;

id_transaccion,COUNT(*)
